# Random Forest experiments

Research questions and hypotheses, written before any forest was trained.

**Main question:** does a Random Forest improve multiclass cyberattack classification over the single classification tree, and which part of the ensemble produces the gain?

**RQ1 - Does the forest beat the single tree, and on which classes?**
Bagging reduces variance and has little effect on bias. I expect macro-F1 to rise, mainly on classes where the tree is unstable. Classes the tree cannot separate at all (Backdoor, Analysis, DoS share feature vectors with other classes) should gain little.

**RQ2 - How does the number of trees affect performance and cost?**
I expect performance to improve until the out-of-bag error stops reducing and then plateau. More trees should not overfit, only cost training time.

**RQ3 - Does random feature selection add anything beyond bagging?**
Subspace sampling adds diversity between trees, so I expect max_features below 1.0 to beat plain bagging (max_features = 1.0). The lecture notes say a forest is generally insensitive to the number of random features, so I expect a flat curve for small values.

**RQ4 - How do tree size and bag size affect the forest?**
Trees grown until the leaves are pure overfit, which is desirable for bagging. I expect unrestricted trees to do well and the pruning the single tree needed to no longer help. Smaller bags should give more diverse but weaker trees and need more trees.

**RQ5 - Which imbalance strategy gives the best balance between overall and minority-class performance?**
I expect balancing inside each tree's bootstrap sample to compete with 5x oversampling at lower cost. Gains on minority classes may cost some precision on Normal and Generic.

Baselines to beat (5-fold mean macro-F1 on the development set): tree 0.6695, kNN 0.6240.

In [8]:
# setup - load the benchmark table and build the 5 encoded folds once
# no oversampling here, the imbalance strategy is one of the things being tested
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, accuracy_score, balanced_accuracy_score
from imblearn.over_sampling import RandomOverSampler

BASE = '/content/drive/MyDrive/Colab Notebooks/Machine Learning/Assignment 4/'
DATA_DIR = BASE + 'data/'
RESULTS_DIR = BASE + 'results/'
SEED = 42
LABEL = 'label_majority'

vec = pd.read_csv(DATA_DIR + 'networkTraffic_vectors.csv')
dev = vec[vec['fold'] >= 0].reset_index(drop=True)

META_COLS = ['label_first', 'label_majority', 'is_conflict', 'n_rows', 'fold']
TREE_DROP = ['ct_ftp_cmd', 'sloss', 'dloss', 'tcprtt']
NOMINAL_COLS = ['proto', 'state', 'service']

def prep_tree_fold(train, val, drop_cols=TREE_DROP):
    # same preparation as the single tree in notebook 01
    X_train = train.drop(columns=META_COLS + drop_cols).copy()
    X_val = val.drop(columns=META_COLS + drop_cols).copy()
    proto_counts = X_train['proto'].value_counts()
    keep_protos = proto_counts[proto_counts >= 0.01 * len(X_train)].index.tolist()
    X_train['proto'] = X_train['proto'].where(X_train['proto'].isin(keep_protos), 'other')
    X_val['proto'] = X_val['proto'].where(X_val['proto'].isin(keep_protos), 'other')
    X_train = pd.get_dummies(X_train, columns=NOMINAL_COLS)
    X_val = pd.get_dummies(X_val, columns=NOMINAL_COLS)
    X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
    return X_train, X_val

def oversample(X, y, mult=5):
    if mult == 1:
        return X, y
    counts = y.value_counts()
    target = {cls: min(n * mult, counts.max()) for cls, n in counts.items()}
    return RandomOverSampler(sampling_strategy=target, random_state=SEED).fit_resample(X, y)

rf_folds = []
for fold in range(5):
    train, val = dev[dev['fold'] != fold], dev[dev['fold'] == fold]
    X_train, X_val = prep_tree_fold(train, val)
    rf_folds.append((X_train, train[LABEL], X_val, val[LABEL]))
    print(f"fold {fold}: train {X_train.shape}, validation {X_val.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
fold 0: train (98357, 59), validation (24590, 59)
fold 1: train (98357, 60), validation (24590, 60)
fold 2: train (98358, 60), validation (24589, 60)
fold 3: train (98358, 59), validation (24589, 59)
fold 4: train (98358, 60), validation (24589, 60)


In [9]:
import sklearn, imblearn
print("scikit-learn", sklearn.__version__, "| imbalanced-learn", imblearn.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__)

scikit-learn 1.6.1 | imbalanced-learn 0.14.2 | pandas 2.2.3 | numpy 2.1.3


In [10]:
# helper used by every experiment below: runs one forest configuration on the 5 folds,
# records all the measures and timings, and adds the rows to one results log
# entropy is fixed to match the single tree, so the forest differs from the tree
# only by bagging and the random feature subsets

import os
pd.set_option('display.width', 200)

# if results were already saved in an earlier session, load them instead of repeating the runs
LOG_PATH = RESULTS_DIR + 'rf_experiments.csv'
results_log = pd.read_csv(LOG_PATH).to_dict('records') if os.path.exists(LOG_PATH) else []
print(len(results_log), "rows already in the results log")

def run_rf(experiment, rq, setting, mult=5, **params):
    rows = []
    for fold, (X_train, y_train, X_val, y_val) in enumerate(rf_folds):
        X_res, y_res = oversample(X_train, y_train, mult)
        model = RandomForestClassifier(criterion='entropy', n_jobs=-1, random_state=SEED, **params)
        start = time.time()
        model.fit(X_res, y_res)
        fit_time = time.time() - start
        start = time.time()
        y_pred = model.predict(X_val)
        predict_time = time.time() - start

        rows.append({'experiment': experiment, 'rq': rq, 'setting': setting, 'oversample': mult,
                     'params': str(params), 'seed': SEED, 'fold': fold,
                     'macro_f1': f1_score(y_val, y_pred, average='macro'),
                     'accuracy': accuracy_score(y_val, y_pred),
                     'weighted_f1': f1_score(y_val, y_pred, average='weighted'),
                     'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
                     'fit_time': fit_time, 'predict_time': predict_time})
    results_log.extend(rows)
    pd.DataFrame(results_log).to_csv(RESULTS_DIR + 'rf_experiments.csv', index=False)
    return pd.DataFrame(rows)

MEASURES = ['macro_f1', 'accuracy', 'weighted_f1', 'balanced_accuracy', 'fit_time', 'predict_time']

# E0 (RQ1) - default forest: 100 trees, sqrt features per split, trees grown fully
# same data preparation and 5x oversampling as the single tree
if any(r['experiment'] == 'E0' for r in results_log):
    e0 = pd.DataFrame([r for r in results_log if r['experiment'] == 'E0'])
else:
    e0 = run_rf('E0', 'RQ1', 'default forest', n_estimators=100)
print(e0[['fold'] + MEASURES].round(4))
print()
print(e0[MEASURES].agg(['mean', 'std']).round(4))

165 rows already in the results log
   fold  macro_f1  accuracy  weighted_f1  balanced_accuracy  fit_time  predict_time
0     0    0.6680    0.8654       0.8641             0.6298   94.3911        0.5035
1     1    0.6795    0.8653       0.8644             0.6417   80.9536        0.4756
2     2    0.6826    0.8656       0.8642             0.6364   85.1587        0.5063
3     3    0.6701    0.8645       0.8628             0.6357   87.3125        0.4960
4     4    0.7012    0.8657       0.8642             0.6576   84.5683        0.5261

      macro_f1  accuracy  weighted_f1  balanced_accuracy  fit_time  predict_time
mean    0.6803    0.8653       0.8639             0.6402   86.4768        0.5015
std     0.0132    0.0005       0.0006             0.0106    4.9805        0.0183


In [11]:
# extended version of the helper from the cell above
# adds the average macro-F1 of the individual trees, which shows how strong each tree is
# on its own compared with the forest as a whole. also lets a run use a different set of folds

def mean_tree_f1(model, X_val, y_val):
    X_arr = X_val.values.astype(np.float32)
    scores = []
    for tree in model.estimators_:
        pred = model.classes_[tree.predict(X_arr).astype(int)]
        scores.append(f1_score(y_val, pred, average='macro'))
    return np.mean(scores)

def run_rf(experiment, rq, setting, mult=5, folds=None, **params):
    folds = rf_folds if folds is None else folds
    rows = []
    for fold, (X_train, y_train, X_val, y_val) in enumerate(folds):
        X_res, y_res = oversample(X_train, y_train, mult)
        settings = {'criterion': 'entropy', **params}
        model = RandomForestClassifier(n_jobs=-1, random_state=SEED, **settings)
        start = time.time()
        model.fit(X_res, y_res)
        fit_time = time.time() - start
        start = time.time()
        y_pred = model.predict(X_val)
        predict_time = time.time() - start

        rows.append({'experiment': experiment, 'rq': rq, 'setting': setting, 'oversample': mult,
                     'params': str(params), 'seed': SEED, 'fold': fold,
                     'macro_f1': f1_score(y_val, y_pred, average='macro'),
                     'accuracy': accuracy_score(y_val, y_pred),
                     'weighted_f1': f1_score(y_val, y_pred, average='weighted'),
                     'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
                     'mean_tree_f1': mean_tree_f1(model, X_val, y_val) if model.n_estimators <= 100 else np.nan,
                     'fit_time': fit_time, 'predict_time': predict_time})
    results_log.extend(rows)
    pd.DataFrame(results_log).to_csv(RESULTS_DIR + 'rf_experiments.csv', index=False)
    print(f"{experiment} | {setting}: macro-F1 {np.mean([r['macro_f1'] for r in rows]):.4f}")
    return pd.DataFrame(rows)

def summarise(experiments, with_reference=False):
    # with_reference adds the 50-tree default forest from E3 as the comparison row
    df = pd.DataFrame(results_log)
    keep = df['experiment'].isin(experiments)
    if with_reference:
        keep = keep | (df['setting'] == 'max_features=sqrt')
    out = df[keep].groupby('setting', sort=False).agg(macro_f1=('macro_f1', 'mean'), std=('macro_f1', 'std'),
                                                      accuracy=('accuracy', 'mean'),
                                                      balanced_accuracy=('balanced_accuracy', 'mean'),
                                                      mean_tree_f1=('mean_tree_f1', 'mean'),
                                                      fit_time=('fit_time', 'mean'))
    return out.round(4)

In [7]:
# E1 (RQ2) - number of trees
# one 500-tree forest is trained per fold and scored using its first 10, 25, ... 500 trees
# the first n trees of a forest are the same as a forest of n trees with the same seed,
# so this gives the whole curve without training seven separate forests
# fit time for n trees is estimated as a share of the 500-tree fit time

tree_counts = [10, 25, 50, 100, 200, 300, 500]

e1_rows = []
for fold, (X_train, y_train, X_val, y_val) in enumerate(rf_folds):
    X_res, y_res = oversample(X_train, y_train, 5)
    model = RandomForestClassifier(n_estimators=max(tree_counts), criterion='entropy', n_jobs=-1, random_state=SEED)
    start = time.time()
    model.fit(X_res, y_res)
    fit_time = time.time() - start

    # sklearn forests average the class probabilities of the trees, so add them up tree by tree
    X_arr = X_val.values.astype(np.float32)
    proba_sum = np.zeros((len(X_arr), len(model.classes_)))
    for i, tree in enumerate(model.estimators_, start=1):
        proba_sum += tree.predict_proba(X_arr)
        if i in tree_counts:
            y_pred = model.classes_[np.argmax(proba_sum, axis=1)]
            e1_rows.append({'experiment': 'E1', 'rq': 'RQ2', 'n_estimators': i, 'seed': SEED, 'fold': fold,
                            'macro_f1': f1_score(y_val, y_pred, average='macro'),
                            'accuracy': accuracy_score(y_val, y_pred),
                            'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
                            'fit_time_est': fit_time * i / max(tree_counts)})
    print(f"fold {fold} done, fit time {fit_time:.0f}s")

e1_df = pd.DataFrame(e1_rows)
e1_df.to_csv(RESULTS_DIR + 'rf_e1_trees.csv', index=False)
print()
print(e1_df.groupby('n_estimators')[['macro_f1', 'accuracy', 'balanced_accuracy', 'fit_time_est']].agg(['mean', 'std']).round(4))

# out-of-bag error against number of trees
# no oversampling here: a duplicated copy of an out-of-bag row could be in the bag, which would make it look too good
# uses the training part of fold 0, warm_start adds trees to the same forest instead of starting again
X_train, y_train, _, _ = rf_folds[0]
oob_rows = []
oob_model = RandomForestClassifier(n_estimators=25, criterion='entropy', oob_score=True,
                                   warm_start=True, n_jobs=-1, random_state=SEED)
for n in [c for c in tree_counts if c >= 25]:
    oob_model.set_params(n_estimators=n)
    oob_model.fit(X_train, y_train)
    oob_rows.append({'n_estimators': n, 'oob_error': 1 - oob_model.oob_score_})

oob_df = pd.DataFrame(oob_rows)
oob_df.to_csv(RESULTS_DIR + 'rf_e1_oob.csv', index=False)
print()
print(oob_df.round(4))

KeyboardInterrupt: 

In [ ]:
# E2 (RQ5) - imbalance strategy, 100 trees
# the 5x oversampling version is the default forest already run in E0
# balanced weights each class by the inverse of its frequency in the whole training set,
# balanced_subsample does the same but recalculated inside each tree's bootstrap sample

run_rf('E2', 'RQ5', 'no correction', mult=1, n_estimators=100)
run_rf('E2', 'RQ5', 'class_weight balanced', mult=1, n_estimators=100, class_weight='balanced')
run_rf('E2', 'RQ5', 'class_weight balanced_subsample', mult=1, n_estimators=100, class_weight='balanced_subsample')
print()
print(summarise(['E0', 'E2']))

In [ ]:
# E3 (RQ3) - number of features considered at each split, 50 trees to keep the run time down
# 1.0 means every feature is considered at every split, which is plain bagging with no feature randomness
# sqrt is the usual default (about 7 or 8 of the 59 columns), 3 is a value from the lecture notes

for mf in ['sqrt', 3, 0.2, 0.5, 1.0]:
    run_rf('E3', 'RQ3', f'max_features={mf}', n_estimators=50, max_features=mf)
print()
print(summarise(['E3']))

In [ ]:
# E4 (RQ4) - restricting the individual trees, 50 trees
# one restriction at a time against the unrestricted forest (the max_features=sqrt run from E3)
# ccp_alpha=0.0003 is the pruning the single tree needed in notebook 01

e4_settings = [
    ('max_depth=15', {'max_depth': 15}),
    ('max_depth=20', {'max_depth': 20}),
    ('max_depth=30', {'max_depth': 30}),
    ('min_samples_leaf=2', {'min_samples_leaf': 2}),
    ('min_samples_leaf=5', {'min_samples_leaf': 5}),
    ('min_samples_leaf=10', {'min_samples_leaf': 10}),
    ('min_samples_split=5', {'min_samples_split': 5}),
    ('min_samples_split=10', {'min_samples_split': 10}),
    ('min_samples_split=20', {'min_samples_split': 20}),
    ('ccp_alpha=0.00003', {'ccp_alpha': 0.00003}),
    ('ccp_alpha=0.0001', {'ccp_alpha': 0.0001}),
    ('ccp_alpha=0.0003', {'ccp_alpha': 0.0003}),
]
for name, p in e4_settings:
    run_rf('E4', 'RQ4', name, n_estimators=50, **p)
print()
print(summarise(['E4'], with_reference=True))

In [ ]:
# E5 (RQ4) - bag size: the share of the training rows each tree's bootstrap sample draws
# tested at 50 and 200 trees to see whether smaller bags need more trees
# None is the default, a bootstrap sample the same size as the training set

for n in [50, 200]:
    for ms in [0.25, 0.5, None]:
        if n == 50 and ms is None:
            continue   # already run in E3 as max_features=sqrt
        run_rf('E5', 'RQ4', f'max_samples={ms}, trees={n}', n_estimators=n, max_samples=ms)
print()
print(summarise(['E5'], with_reference=True))

# two short checks, 50 trees
# gini instead of entropy, and the two-feature connection-count cluster instead of all seven
CLUSTER_DROP = ['ct_dst_src_ltm', 'ct_src_dport_ltm', 'ct_dst_ltm', 'ct_src_ltm', 'ct_dst_sport_ltm']
rf_folds_2c = []
for fold in range(5):
    train, val = dev[dev['fold'] != fold], dev[dev['fold'] == fold]
    X_train, X_val = prep_tree_fold(train, val, TREE_DROP + CLUSTER_DROP)
    rf_folds_2c.append((X_train, train[LABEL], X_val, val[LABEL]))

run_rf('checks', 'sensitivity', 'gini', n_estimators=50, criterion='gini')
run_rf('checks', 'sensitivity', '2 cluster features', folds=rf_folds_2c, n_estimators=50)
print()
print(summarise(['checks'], with_reference=True))

In [ ]:
# follow-up runs prompted by E3, where max_features=0.5 was clearly better than the default
# these were not in the original plan, they are added because of that result

# 1. the 50% setting with 200 trees, the tree count E1 pointed to
run_rf('E6', 'RQ1', 'max_features=0.5, trees=200', n_estimators=200, max_features=0.5)

# 2. values either side of 0.5 to see how sharp the peak is
for mf in [0.35, 0.65]:
    run_rf('E3b', 'RQ3', f'max_features={mf}', n_estimators=50, max_features=mf)

# 3. stronger oversampling at the 50% setting - 5x was tuned for the single tree
#    and the forest was behind the tree on balanced accuracy
run_rf('E2b', 'RQ5', 'oversampling 10x, max_features=0.5', mult=10, n_estimators=50, max_features=0.5)

print()
print(summarise(['E3', 'E3b']))
print()
print(summarise(['E2b', 'E6']))

In [ ]:
# one more oversampling ratio, since 10x beat 5x and the best value might be higher still
# skipped if it is already in the results log from an earlier session
import gc
if not any(r['setting'] == 'oversampling 20x, max_features=0.5' for r in results_log):
    run_rf('E2b', 'RQ5', 'oversampling 20x, max_features=0.5', mult=20, n_estimators=50, max_features=0.5)

log_df = pd.DataFrame(results_log)
os_settings = {5: 'max_features=0.5', 10: 'oversampling 10x, max_features=0.5', 20: 'oversampling 20x, max_features=0.5'}
os_means = {m: log_df[log_df['setting'] == s]['macro_f1'].mean() for m, s in os_settings.items()}
os_stds = {m: log_df[log_df['setting'] == s]['macro_f1'].std() for m, s in os_settings.items()}
print({k: round(float(v), 4) for k, v in os_means.items()})

# one-standard-error rule, same as for the single tree: the smallest ratio that is
# within one standard error of the best. more oversampling costs time and memory
best_mult = max(os_means, key=os_means.get)
threshold = os_means[best_mult] - os_stds[best_mult] / np.sqrt(5)
FINAL_MULT = min(m for m in os_means if os_means[m] >= threshold)
print(f"Best mean at {best_mult}x, threshold {threshold:.4f}, chosen oversampling ratio: {FINAL_MULT}")

# final forest on the development folds
# 200 trees (E1), half the features at each split (E3), unrestricted trees (E4), default bag size (E5)
# results are saved after every fold, and folds already saved are not repeated
FINAL_PARAMS = {'n_estimators': 200, 'max_features': 0.5, 'criterion': 'entropy'}
FINAL_PATH = RESULTS_DIR + 'final_forest_cv.csv'
PRED_PATH = RESULTS_DIR + 'final_forest_cv_preds.csv'

rf_results, rf_preds = [], []
if os.path.exists(FINAL_PATH) and os.path.exists(PRED_PATH):
    saved = pd.read_csv(FINAL_PATH)
    # only reuse saved folds if they were run with the same oversampling ratio
    if 'oversample' in saved.columns and (saved['oversample'] == FINAL_MULT).all():
        rf_results = saved.to_dict('records')
        rf_preds = [pd.read_csv(PRED_PATH)]
done_folds = {r['fold'] for r in rf_results}

for fold, (X_train, y_train, X_val, y_val) in enumerate(rf_folds):
    if fold in done_folds:
        print(f"fold {fold} already saved")
        continue
    X_res, y_res = oversample(X_train, y_train, FINAL_MULT)
    model = RandomForestClassifier(n_jobs=-1, random_state=SEED, **FINAL_PARAMS)
    start = time.time()
    model.fit(X_res, y_res)
    fit_time = time.time() - start
    start = time.time()
    y_pred = model.predict(X_val)
    predict_time = time.time() - start

    rf_results.append({'model': 'forest', 'fold': fold, 'oversample': FINAL_MULT,
                       'macro_f1': f1_score(y_val, y_pred, average='macro'),
                       'accuracy': accuracy_score(y_val, y_pred),
                       'weighted_f1': f1_score(y_val, y_pred, average='weighted'),
                       'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
                       'fit_time': fit_time, 'predict_time': predict_time})
    rf_preds.append(pd.DataFrame({'fold': fold, 'y_true': y_val.values, 'y_pred': y_pred}))
    pd.DataFrame(rf_results).to_csv(FINAL_PATH, index=False)
    pd.concat(rf_preds).to_csv(PRED_PATH, index=False)
    print(f"fold {fold} done, macro-F1 {rf_results[-1]['macro_f1']:.4f}")

    # free the forest and the resampled data before the next fold
    del model, X_res, y_res
    gc.collect()

rf_results_df = pd.DataFrame(rf_results).sort_values('fold').reset_index(drop=True)

# all three models side by side on the same development folds
all_models = pd.concat([pd.read_csv(RESULTS_DIR + 'baseline_knn_cv.csv'),
                        pd.read_csv(RESULTS_DIR + 'baseline_tree_cv.csv'),
                        rf_results_df.drop(columns='oversample')])
print()
print(rf_results_df.round(4))
print()
print(all_models.drop(columns='fold').groupby('model', sort=False).agg(['mean', 'std']).round(4).T)

In [13]:
# E7 - boosting comparison (not in the original plan)
# bagging was chosen over boosting because of the label noise, but that was only argued from
# the literature. this cell tests it: AdaBoost with entropy trees, same folds,
# same data preparation and same 5x oversampling as the single tree
# the depth of the member trees is tuned on all five folds, like every other parameter
# staged_predict gives the score after 10, 25 and 50 rounds from one fit
# AdaBoost cannot train its trees in parallel, so 50 rounds is the limit for run time
import os, warnings
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
warnings.filterwarnings('ignore', message='X does not have valid feature names')

BOOST_PATH = RESULTS_DIR + 'boosting_cv.csv'
boost_depths = [6, 10, 15]
boost_rounds = [10, 25, 50]
boost_rows = pd.read_csv(BOOST_PATH).to_dict('records') if os.path.exists(BOOST_PATH) else []

for depth in boost_depths:
    for fold, (X_train, y_train, X_val, y_val) in enumerate(rf_folds):
        if any(r['max_depth'] == depth and r['fold'] == fold for r in boost_rows):
            print(f"depth {depth}, fold {fold} already saved")
            continue
        X_res, y_res = oversample(X_train, y_train, 5)
        base = DecisionTreeClassifier(criterion='entropy', max_depth=depth, random_state=SEED)
        model = AdaBoostClassifier(estimator=base, n_estimators=max(boost_rounds), random_state=SEED)
        start = time.time()
        model.fit(X_res, y_res)
        fit_time = time.time() - start

        # training score as well, to compare with the forest and the single tree
        train_f1 = f1_score(y_train, model.predict(X_train), average='macro')
        for i, y_pred in enumerate(model.staged_predict(X_val), start=1):
            if i in boost_rounds:
                boost_rows.append({'model': 'adaboost', 'max_depth': depth, 'n_estimators': i, 'fold': fold,
                                   'macro_f1': f1_score(y_val, y_pred, average='macro'),
                                   'accuracy': accuracy_score(y_val, y_pred),
                                   'balanced_accuracy': balanced_accuracy_score(y_val, y_pred),
                                   'train_macro_f1': train_f1 if i == max(boost_rounds) else np.nan,
                                   'fit_time': fit_time * i / max(boost_rounds)})
        # saved after every fold so a disconnect does not lose the finished folds
        pd.DataFrame(boost_rows).to_csv(BOOST_PATH, index=False)
        print(f"depth {depth}, fold {fold} done in {fit_time:.0f}s, macro-F1 {boost_rows[-1]['macro_f1']:.4f}")

boost_df = pd.DataFrame(boost_rows)
print()
print(boost_df.groupby(['max_depth', 'n_estimators'])[['macro_f1', 'accuracy', 'balanced_accuracy',
                                                         'train_macro_f1', 'fit_time']].agg(['mean', 'std']).round(4))
print()
print("for comparison: single tree 0.6695, default forest 0.6803, final forest 0.7354")

depth 6, fold 0 done in 251s, macro-F1 0.5838
depth 6, fold 1 done in 246s, macro-F1 0.5988
depth 6, fold 2 done in 247s, macro-F1 0.5955
depth 6, fold 3 done in 245s, macro-F1 0.5939
depth 6, fold 4 done in 247s, macro-F1 0.6187
depth 10, fold 0 done in 361s, macro-F1 0.6797
depth 10, fold 1 done in 360s, macro-F1 0.6497
depth 10, fold 2 done in 362s, macro-F1 0.6606
depth 10, fold 3 done in 361s, macro-F1 0.6490
depth 10, fold 4 done in 363s, macro-F1 0.6803
depth 15, fold 0 done in 518s, macro-F1 0.6531
depth 15, fold 1 done in 513s, macro-F1 0.6657
depth 15, fold 2 done in 517s, macro-F1 0.6662
depth 15, fold 3 done in 513s, macro-F1 0.6357
depth 15, fold 4 done in 523s, macro-F1 0.6583

                       macro_f1         accuracy         balanced_accuracy         train_macro_f1          fit_time        
                           mean     std     mean     std              mean     std           mean     std      mean     std
max_depth n_estimators                             

In [12]:
print(len(pd.read_csv(RESULTS_DIR + 'rf_experiments.csv')))

165
